# NSFG Analysis Report (Updated)
Includes all models and corrections.

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

# Load cleaned data
df = pd.read_csv("nsfg_cleaned.csv")
df['CURRMETH1'] = df['CURRMETH1'].astype("category")
df['AGE_GROUP'] = pd.cut(df['AGE_R'], bins=[14, 19, 24, 29, 34, 39, 44, 49, 54],
                         labels=['15–19', '20–24', '25–29', '30–34', '35–39', '40–44', '45–49', '50–54'])


## Poisson Regression

In [ ]:

df_pois = df.dropna(subset=['PREGNUM', 'AGE_GROUP', 'RSCRRACE', 'RSCRHISP', 'SMK100', 'DRINK12', 'MARSTAT'])
formula = "PREGNUM ~ C(AGE_GROUP) + C(RSCRRACE) + C(RSCRHISP) + SMK100 + DRINK12 + C(MARSTAT)"
poisson_model = smf.glm(formula=formula, data=df_pois, family=sm.families.Poisson()).fit()
print(poisson_model.summary())


## Contingency Table: CURRMETH1 × MARSTAT

In [ ]:

contingency = pd.crosstab(df['CURRMETH1'], df['MARSTAT'])
chi2, p, dof, exp = chi2_contingency(contingency)
print("Chi-square test p-value:", p)


## Ordinal Logistic Regression

In [ ]:

df_ord = df.dropna(subset=["GENHEALT", "AGE_R", "NUMCHILD", "SMK100", "DRINK12"])
df_ord["GENHEALT"] = df_ord["GENHEALT"].astype(int)
ord_model = OrderedModel(df_ord["GENHEALT"],
                         df_ord[["AGE_R", "NUMCHILD", "SMK100", "DRINK12"]],
                         distr="logit")
ord_res = ord_model.fit(method="bfgs")
print(ord_res.summary())


## Multinomial Logistic Regression

In [ ]:

df_mnlogit = df.dropna(subset=["CURRMETH1"])
mnlogit_model = smf.mnlogit("CURRMETH1 ~ AGE_R + NUMCHILD + C(MARSTAT)", data=df_mnlogit).fit()
print(mnlogit_model.summary())


## EDA and Multiple Comparisons

In [ ]:

sns.boxplot(x="RSCRRACE", y="PREGNUM", data=df)
plt.title("PREGNUM by RSCRRACE")
plt.tight_layout()
plt.savefig("pregnum_by_race.png")
plt.show()

anova_model = smf.ols("PREGNUM ~ C(RSCRRACE)", data=df).fit()
anova_result = sm.stats.anova_lm(anova_model, typ=2)
print(anova_result)

race_levels = df["RSCRRACE"].dropna().unique()
pvals = []
for i, r1 in enumerate(race_levels):
    for r2 in race_levels[i+1:]:
        g1 = df[df["RSCRRACE"] == r1]["PREGNUM"]
        g2 = df[df["RSCRRACE"] == r2]["PREGNUM"]
        _, pval, _ = sm.stats.ttest_ind(g1, g2)
        pvals.append(pval)
_, bonf_pvals, _, _ = multipletests(pvals, method='bonferroni')
print("Bonferroni corrected p-values:", bonf_pvals)
